
# 03 - Silver Layer: Data Transformation
# Clean and Transform MLB Raw Data into Structured Table


In [0]:
# Imports
import json
import uuid
import traceback

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from pyspark.sql import Row
from pyspark.sql.types import *

In [0]:
# Configuración
pipeline_name = "mlb_daily_games_etl"
source_api = "MLB Stats API"
timezone = "America/Santo_Domingo"

pipeline_run_id = str(uuid.uuid4())
current_time_rd = datetime.now(ZoneInfo(timezone))

execution_date = (current_time_rd - timedelta(days=1)).strftime("%Y-%m-%d")

print(f"Pipeline Run ID: {pipeline_run_id}")
print(f"Execution Date: {execution_date}")

In [0]:
# Función de logging corregida
def log_event(layer, status, message, records_processed, started_at):
    ended_at = datetime.now(ZoneInfo(timezone))
    duration_seconds = float((ended_at - started_at).total_seconds())

    log_schema = StructType([
        StructField("pipeline_name", StringType(), True),
        StructField("pipeline_run_id", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
        StructField("execution_date", StringType(), True),
        StructField("records_processed", IntegerType(), True),
        StructField("started_at", TimestampType(), True),
        StructField("ended_at", TimestampType(), True),
        StructField("duration_seconds", DoubleType(), True)
    ])

    log_row = [(
        pipeline_name,
        pipeline_run_id,
        layer,
        status,
        message,
        execution_date,
        int(records_processed),
        started_at,
        ended_at,
        duration_seconds
    )]

    log_df = spark.createDataFrame(log_row, schema=log_schema)

    log_df.write.format("delta").mode("append").saveAsTable(
        "mlb_project.pipeline_logs"
    )

In [0]:
# Leer datos desde Bronze
silver_started_at = datetime.now(ZoneInfo(timezone))

try:
    bronze_df = spark.sql(f"""
    SELECT raw_json
    FROM mlb_project.bronze_mlb_schedule_raw
    WHERE execution_date = '{execution_date}'
    ORDER BY extracted_at DESC
    LIMIT 1
    """)

    if bronze_df.count() == 0:
        raise Exception(f"No Bronze data found for date: {execution_date}")

    raw_json = bronze_df.collect()[0]["raw_json"]
    raw_data = json.loads(raw_json)

    print("Bronze data loaded successfully.")

except Exception:
    error_message = traceback.format_exc()

    log_event(
        layer="silver",
        status="FAILED",
        message=error_message,
        records_processed=0,
        started_at=silver_started_at
    )

    raise Exception(error_message)

In [0]:
#Transformar JSON
try:
    games_list = []

    for date_block in raw_data.get("dates", []):
        game_date = date_block.get("date")

        for game in date_block.get("games", []):
            teams = game.get("teams", {})

            home = teams.get("home", {})
            away = teams.get("away", {})

            home_team = home.get("team", {})
            away_team = away.get("team", {})

            venue = game.get("venue", {})
            status = game.get("status", {})
            linescore = game.get("linescore", {})

            games_list.append(Row(
                game_pk=game.get("gamePk"),
                game_date=game_date,
                official_date=game.get("officialDate"),
                season=game.get("season"),
                game_type=game.get("gameType"),
                abstract_game_state=status.get("abstractGameState"),
                game_status=status.get("detailedState"),
                home_team_id=home_team.get("id"),
                home_team_name=home_team.get("name"),
                away_team_id=away_team.get("id"),
                away_team_name=away_team.get("name"),
                home_score=home.get("score"),
                away_score=away.get("score"),
                venue_id=venue.get("id"),
                venue_name=venue.get("name"),
                current_inning=linescore.get("currentInning"),
                inning_state=linescore.get("inningState"),
                is_final=True if status.get("abstractGameState") == "Final" else False,
                source_api=source_api,
                pipeline_run_id=pipeline_run_id,
                extracted_at=datetime.now(ZoneInfo(timezone)),
                updated_at=datetime.now(ZoneInfo(timezone))
            ))

    print(f"Games transformed: {len(games_list)}")

except Exception:
    error_message = traceback.format_exc()

    log_event(
        layer="silver",
        status="FAILED",
        message=error_message,
        records_processed=0,
        started_at=silver_started_at
    )

    raise Exception(error_message)

In [0]:
# Crear DataFrame Silver
silver_schema = StructType([
    StructField("game_pk", LongType(), True),
    StructField("game_date", StringType(), True),
    StructField("official_date", StringType(), True),
    StructField("season", StringType(), True),
    StructField("game_type", StringType(), True),
    StructField("abstract_game_state", StringType(), True),
    StructField("game_status", StringType(), True),
    StructField("home_team_id", IntegerType(), True),
    StructField("home_team_name", StringType(), True),
    StructField("away_team_id", IntegerType(), True),
    StructField("away_team_name", StringType(), True),
    StructField("home_score", IntegerType(), True),
    StructField("away_score", IntegerType(), True),
    StructField("venue_id", IntegerType(), True),
    StructField("venue_name", StringType(), True),
    StructField("current_inning", IntegerType(), True),
    StructField("inning_state", StringType(), True),
    StructField("is_final", BooleanType(), True),
    StructField("source_api", StringType(), True),
    StructField("pipeline_run_id", StringType(), True),
    StructField("extracted_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

silver_df = spark.createDataFrame(games_list, schema=silver_schema)

display(silver_df)

In [0]:
# MERGE hacia Silver
try:
    silver_df.createOrReplaceTempView("silver_updates")

    spark.sql("""
    MERGE INTO mlb_project.silver_mlb_games AS target
    USING silver_updates AS source
    ON target.game_pk = source.game_pk

    WHEN MATCHED THEN UPDATE SET
        target.game_date = source.game_date,
        target.official_date = source.official_date,
        target.season = source.season,
        target.game_type = source.game_type,
        target.abstract_game_state = source.abstract_game_state,
        target.game_status = source.game_status,
        target.home_team_id = source.home_team_id,
        target.home_team_name = source.home_team_name,
        target.away_team_id = source.away_team_id,
        target.away_team_name = source.away_team_name,
        target.home_score = source.home_score,
        target.away_score = source.away_score,
        target.venue_id = source.venue_id,
        target.venue_name = source.venue_name,
        target.current_inning = source.current_inning,
        target.inning_state = source.inning_state,
        target.is_final = source.is_final,
        target.source_api = source.source_api,
        target.pipeline_run_id = source.pipeline_run_id,
        target.updated_at = source.updated_at

    WHEN NOT MATCHED THEN INSERT *
    """)

    log_event(
        layer="silver",
        status="SUCCESS",
        message="Silver MERGE completed successfully.",
        records_processed=silver_df.count(),
        started_at=silver_started_at
    )

    print("Silver MERGE completed successfully.")

except Exception:
    error_message = traceback.format_exc()

    log_event(
        layer="silver",
        status="FAILED",
        message=error_message,
        records_processed=0,
        started_at=silver_started_at
    )

    raise Exception(error_message)

In [0]:
# Validar Silver
display(
    spark.sql("""
    SELECT *
    FROM mlb_project.silver_mlb_games
    ORDER BY game_date DESC, home_team_name
    LIMIT 20
    """)
)